In [87]:
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("HUGGINGFACEHUB_API_TOKEN")

## Install libraries

In [88]:
#!pip install -q youtube-transcript-api langchain-community langchain-text-splitters langchain-openai \
#               faiss-cpu tiktoken python-dotenv

In [89]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

## Step 1a - Indexing (Document Ingestion)

In [90]:
video_id = "HJ0dkXd1-lE"
from youtube_transcript_api import YouTubeTranscriptApi

ytt_api = YouTubeTranscriptApi()
ytt_api.fetch(video_id)

FetchedTranscript(snippets=[FetchedTranscriptSnippet(text='This cockroach party phenomena is', start=0.16, duration=5.199), FetchedTranscriptSnippet(text='getting more interesting by the day.', start=2.0, duration=7.08), FetchedTranscriptSnippet(text='More or less successful', start=5.359, duration=3.721), FetchedTranscriptSnippet(text='is', start=11.28, duration=3.0), FetchedTranscriptSnippet(text='maybe the numbers were not as good as', start=14.32, duration=4.08), FetchedTranscriptSnippet(text='they had desired. But let me talk about', start=16.32, duration=4.799), FetchedTranscriptSnippet(text='the symbolism. Dr. Ambedkar Jabe the', start=18.4, duration=5.6), FetchedTranscriptSnippet(text='book in his hands and this jab movement.', start=21.119, duration=5.441), FetchedTranscriptSnippet(text='Why? It was not random. It was not as if', start=24.0, duration=4.16), FetchedTranscriptSnippet(text="he was passing time. I don't believe", start=26.56, duration=3.6), FetchedTranscriptSnippe

In [91]:
ytt_api = YouTubeTranscriptApi()
fetched_transcript = ytt_api.fetch(video_id)

# is iterable
for snippet in fetched_transcript:
    print(snippet.text)

# indexable
last_snippet = fetched_transcript[-1]

# provides a length
snippet_count = len(fetched_transcript)

This cockroach party phenomena is
getting more interesting by the day.
More or less successful
is
maybe the numbers were not as good as
they had desired. But let me talk about
the symbolism. Dr. Ambedkar Jabe the
book in his hands and this jab movement.
Why? It was not random. It was not as if
he was passing time. I don't believe
that he was passing time reading Dr.
Ambedkar's biography on the plane. It
was strategically placed there. He
wanted it there and no problem with
that. No harm in it. The symbolism is
very critical. Dr. Ambedkar stood for
dignity. He stood for dignity through
education. He stood against oppression
of marginalized. He was a champion of
the marginalized. And that symbolism to
borrow that symbolism is genius. He
stood against cast oppression. Even
though there is no explicit cast issue
ever stated so far in this thing but
there was an undercurrent notice that
had the castish sort of marginalized
oppressed dignity element in it. I'm not
saying that it was a cast h

In [92]:
fetched_transcript.to_raw_data()

[{'text': 'This cockroach party phenomena is',
  'start': 0.16,
  'duration': 5.199},
 {'text': 'getting more interesting by the day.',
  'start': 2.0,
  'duration': 7.08},
 {'text': 'More or less successful', 'start': 5.359, 'duration': 3.721},
 {'text': 'is', 'start': 11.28, 'duration': 3.0},
 {'text': 'maybe the numbers were not as good as',
  'start': 14.32,
  'duration': 4.08},
 {'text': 'they had desired. But let me talk about',
  'start': 16.32,
  'duration': 4.799},
 {'text': 'the symbolism. Dr. Ambedkar Jabe the',
  'start': 18.4,
  'duration': 5.6},
 {'text': 'book in his hands and this jab movement.',
  'start': 21.119,
  'duration': 5.441},
 {'text': 'Why? It was not random. It was not as if',
  'start': 24.0,
  'duration': 4.16},
 {'text': "he was passing time. I don't believe",
  'start': 26.56,
  'duration': 3.6},
 {'text': 'that he was passing time reading Dr.',
  'start': 28.16,
  'duration': 4.48},
 {'text': "Ambedkar's biography on the plane. It",
  'start': 30.16,
 

## Step 1b - Indexing (Text Splitting)

In [93]:
#pip install -q textblob

In [94]:
# Create a text blob from the fetched transcript snippets
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
transcript_text = " ".join([s["text"] for s in fetched_transcript.to_raw_data()])
chunks = splitter.create_documents([transcript_text])

In [95]:
# Create a text blob from the fetched transcript snippets
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

# FIX: Add () to call the method
transcript_text = " ".join([s["text"] for s in fetched_transcript.to_raw_data()])
chunks = splitter.create_documents([transcript_text])

In [96]:
len(chunks)

3

## Step 1c & 1d - Indexing (Embedding Generation and Storing in Vector Store)

In [97]:
from langchain_huggingface import HuggingFaceEndpointEmbeddings

In [98]:
embedding = HuggingFaceEndpointEmbeddings(repo_id='sentence-transformers/all-MiniLM-L6-v2')

In [99]:
#embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vector_store = FAISS.from_documents(chunks, embedding)

In [100]:
vector_store.index_to_docstore_id

{0: 'd466d668-3904-426b-b1cb-e5ea3ba6d45e',
 1: 'be61b07b-5bf9-4641-9dbe-c47671f1da89',
 2: 'ac5a227e-b9b2-4912-a318-82c030c225b0'}

In [101]:
vector_store.get_by_ids(['746d9727-66d4-4e6d-adef-f1a6b08859f5'])

[]

## Step 2 - Retrieval

In [102]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [103]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEndpointEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001E3DC471C60>, search_kwargs={'k': 4})

In [104]:
retriever.invoke('What is deepmind')

[Document(id='ac5a227e-b9b2-4912-a318-82c030c225b0', metadata={}, page_content='They will want jobs which is the primary issue. the lack of good jobs. That is where dignity really comes into the picture if the government is listening and if they are observing carefully. Right now the issue remains constrained to this institutional mistrust. But the moment you start seeing protests like this for jobs that is where the government is in serious trouble and that is where I would say that this movement has now achieved maturity.'),
 Document(id='d466d668-3904-426b-b1cb-e5ea3ba6d45e', metadata={}, page_content="This cockroach party phenomena is getting more interesting by the day. More or less successful is maybe the numbers were not as good as they had desired. But let me talk about the symbolism. Dr. Ambedkar Jabe the book in his hands and this jab movement. Why? It was not random. It was not as if he was passing time. I don't believe that he was passing time reading Dr. Ambedkar's biograp

## Step 3 - Augmentation

In [105]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.messages import HumanMessage


In [106]:
llm = HuggingFaceEndpoint(
    repo_id="deepseek-ai/DeepSeek-V4-Pro",
    task="conversational",
    provider="novita"
)

In [107]:
chat_model = ChatHuggingFace(llm=llm)

In [108]:
# llm = HuggingFaceEndpoint(
#     repo_id="deepseek-ai/DeepSeek-V4-Pro", #change from deepseek-ai/DeepSeek-V4-Pro
#     task="text-generation" #change from text-generation
#     # provider="novita"
# )

In [109]:
#llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)

In [110]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [111]:
question          = "is the topic of nuclear fusion discussed in this video? if yes then what was discussed"
retrieved_docs    = retriever.invoke(question)

In [112]:
retrieved_docs

[Document(id='d466d668-3904-426b-b1cb-e5ea3ba6d45e', metadata={}, page_content="This cockroach party phenomena is getting more interesting by the day. More or less successful is maybe the numbers were not as good as they had desired. But let me talk about the symbolism. Dr. Ambedkar Jabe the book in his hands and this jab movement. Why? It was not random. It was not as if he was passing time. I don't believe that he was passing time reading Dr. Ambedkar's biography on the plane. It was strategically placed there. He wanted it there and no problem with that. No harm in it. The symbolism is very critical. Dr. Ambedkar stood for dignity. He stood for dignity through education. He stood against oppression of marginalized. He was a champion of the marginalized. And that symbolism to borrow that symbolism is genius. He stood against cast oppression. Even though there is no explicit cast issue ever stated so far in this thing but there was an undercurrent notice that had the castish sort of m

In [113]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

"This cockroach party phenomena is getting more interesting by the day. More or less successful is maybe the numbers were not as good as they had desired. But let me talk about the symbolism. Dr. Ambedkar Jabe the book in his hands and this jab movement. Why? It was not random. It was not as if he was passing time. I don't believe that he was passing time reading Dr. Ambedkar's biography on the plane. It was strategically placed there. He wanted it there and no problem with that. No harm in it. The symbolism is very critical. Dr. Ambedkar stood for dignity. He stood for dignity through education. He stood against oppression of marginalized. He was a champion of the marginalized. And that symbolism to borrow that symbolism is genius. He stood against cast oppression. Even though there is no explicit cast issue ever stated so far in this thing but there was an undercurrent notice that had the castish sort of marginalized oppressed dignity element in it. I'm not saying that it was a cast\

In [114]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

In [115]:
final_prompt

StringPromptValue(text="\n      You are a helpful assistant.\n      Answer ONLY from the provided transcript context.\n      If the context is insufficient, just say you don't know.\n\n      This cockroach party phenomena is getting more interesting by the day. More or less successful is maybe the numbers were not as good as they had desired. But let me talk about the symbolism. Dr. Ambedkar Jabe the book in his hands and this jab movement. Why? It was not random. It was not as if he was passing time. I don't believe that he was passing time reading Dr. Ambedkar's biography on the plane. It was strategically placed there. He wanted it there and no problem with that. No harm in it. The symbolism is very critical. Dr. Ambedkar stood for dignity. He stood for dignity through education. He stood against oppression of marginalized. He was a champion of the marginalized. And that symbolism to borrow that symbolism is genius. He stood against cast oppression. Even though there is no explicit 

## Step 4 - Generation

In [116]:
answer = chat_model.invoke(final_prompt)
print(answer.content)

No, the topic of nuclear fusion is not discussed in this video. The transcript focuses on symbolism related to Dr. Ambedkar, dignity, oppression, exam mismanagement, institutional mistrust, hyper-competition, and the lack of good jobs.


## Building a Chain

In [117]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [118]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [119]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [120]:
parallel_chain.invoke('who is Demis')

{'context': "This cockroach party phenomena is getting more interesting by the day. More or less successful is maybe the numbers were not as good as they had desired. But let me talk about the symbolism. Dr. Ambedkar Jabe the book in his hands and this jab movement. Why? It was not random. It was not as if he was passing time. I don't believe that he was passing time reading Dr. Ambedkar's biography on the plane. It was strategically placed there. He wanted it there and no problem with that. No harm in it. The symbolism is very critical. Dr. Ambedkar stood for dignity. He stood for dignity through education. He stood against oppression of marginalized. He was a champion of the marginalized. And that symbolism to borrow that symbolism is genius. He stood against cast oppression. Even though there is no explicit cast issue ever stated so far in this thing but there was an undercurrent notice that had the castish sort of marginalized oppressed dignity element in it. I'm not saying that it

In [121]:
parser = StrOutputParser()

In [122]:
main_chain = parallel_chain | prompt | chat_model | parser

In [ ]:
main_chain.invoke('Can you summarize the video')

'The speaker discusses a protest movement, noting that while turnout may not have met organizers’ hopes, the strategic use of Dr. Ambedkar’s symbolism is significant. Ambedkar represents dignity through education, opposition to caste oppression, and championing the marginalized. The speaker argues this symbolism borrows dignity against privilege, but the core issue isn’t just privilege—it’s a mix of three problems: exam mismanagement and institutional mistrust (the immediate plank), hyper-competition, and ultimately the lack of good jobs. The speaker warns that if protests shift from institutional mistrust to directly demanding jobs, the government will face serious trouble, and such a shift would signal the movement’s maturity.'

: 